In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/IPL.csv.xls')
print('Shape:', df.shape)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df = df[df['innings'] == 2].copy()
df = df.dropna(subset=['runs_target'])
df = df[df['city'] != 'Unknown']
print('Rows remaining:', len(df))

In [ ]:
# Dictionary mapping old names → new names
team_rename = {
 'Delhi Daredevils'
: 'Delhi Capitals',
'Deccan Chargers'
# renamed in 2019
 : 'Sunrisers Hyderabad',   # replaced in 2013
 'Kings XI Punjab'


: 'Punjab Kings',
# renamed in 2021
 'Rising Pune Supergiant'
 : 'Rising Pune Supergiants', # name fix
 'Royal Challengers Bangalore' : 'Royal Challengers Bengaluru', # renamed in 2023
 }
 # Apply the rename to all three team-related columns
 # .replace() swaps old values with new values based on the dictionary
for col in ['batting_team', 'bowling_team', 'match_won_by']:
  df[col] = df[col].replace(team_rename)


In [ ]:
 # List of current active IPL franchises (10 teams)
 TEAMS = ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans',
 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians',
 'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bengaluru',
 'Sunrisers Hyderabad']
 # .isin(TEAMS) returns True only if the value is in our TEAMS list
 # We apply this to BOTH batting and bowling team columns using &  (AND)
 # This removes defunct teams like Kochi Tuskers, Pune Warriors etc.
 df = df[df['batting_team'].isin(TEAMS) & df['bowling_team'].isin(TEAMS)]
 # Verify — should show exactly 10 teams
 print('Teams:', sorted(df['batting_team'].unique()))

In [ ]:
# Remove rows where team_balls = 0 (the very first ball of the innings)
# We can't divide by 0 when calculating run rates
df = df[df['team_balls'] > 0]
# How many more runs does the batting team need to win?
df['runs_left'] = df['runs_target'] - df['team_runs']
# How many valid balls are remaining? (T20 = 120 balls total)
# team_balls counts only legal deliveries (not wides/no-balls)
df['balls_left'] = 120 - df['team_balls']
# How many wickets does the batting team still have?
df['wickets_remaining'] = 10 - df['team_wicket']
# Current Run Rate = runs scored so far per over
# (team_balls / 6) converts balls to overs
df['crr'] = (df['team_runs'] * 6 / df['team_balls']).round(2)
# Required Run Rate = runs still needed per over remaining
df['rrr'] = (df['runs_left'] * 6 / df['balls_left']).round(2)
# Target variable: 1 if the batting team won, 0 if they lost
# This is what the model will learn to predict
df['result'] = (df['batting_team'] == df['match_won_by']).astype(int)

In [ ]:
 # Select only the columns we need — 9 features + 1 target
 FEATURES = ['batting_team', 'bowling_team', 'city',
 'runs_left', 'balls_left', 'wickets_remaining',
 'runs_target', 'crr', 'rrr', 'result']
 # Create a clean copy with only these columns
 final_df = df[FEATURES].copy()
 # Replace infinity values (happen when dividing by zero) with NaN
 # Then drop any rows that have NaN in any column
 # np.inf = positive infinity, -np.inf = negative infinity
 final_df = final_df.replace([np.inf, -np.inf], np.nan).dropna()
 # Confirm the final shape of our clean dataset
 print('Final shape:', final_df.shape)
 final_df.head()

In [ ]:
# Calculate win percentage for each batting team
win_pct = final_df.groupby('batting_team')['result'].mean() * 100
win_pct = win_pct.sort_values(ascending=False)

# .plot(kind='bar') creates a bar chart from the Series
win_pct.plot(kind='bar', figsize=(11, 4), color='steelblue')
# Add a title to the chart
plt.title('Win % While Chasing — By Team')
# Draw a horizontal red dashed line at 50% for reference
plt.axhline(50, color='red', linestyle='--', label='50%')
# Rotate x-axis labels so team names don't overlap
plt.xticks(rotation=40, ha='right')
# Automatically adjust spacing so nothing gets cut off
plt.tight_layout()
# Display the chart
plt.show()